# SegFormer-B0 — Microscopy Semantic Segmentation

**Research code · Dr. Gülsüm Meço**

Original research implementation for SegFormer-B0 semantic segmentation of microscopy imagery. Includes COCO polygon-to-mask preparation, model configuration, training, foreground metrics, early stopping and checkpoint saving.

**Prerequisites:** Requires an externally provided COCO microscopy dataset with train/valid/test splits at /content/own_polygonal_39 and a Google Drive mount for the configured checkpoint path.

**Dataset classes:** background, onion, seed, stem. Dataset images, annotation files, and trained model weights are **not published** in this repository.

**Release note:** Adapted from the author's working Colab notebook without including its exploratory cells or execution outputs. This notebook preserves original experimental code but was not re-run end to end as part of the public-release preparation.


In [ ]:
# ============================================================
# OWN_POLYGONAL_39 — SEGFORMER-B0 RECONSTRUCTION
# ============================================================

!pip -q install pycocotools transformers

import os, json, random
import numpy as np
from PIL import Image
from collections import defaultdict

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from pycocotools import mask as maskUtils
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# ------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------
SEED = 42
BASE = "/content/own_polygonal_39"
MASK_BASE = "/content/own_polygonal_39_masks"

LOCAL_SAVE = "/content/best_segformer_own_polygonal_39.pt"
DRIVE_SAVE = "/content/drive/MyDrive/own_polygonal_39_checkpoints/best_segformer_own_polygonal_39.pt"

MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"

NUM_CLASSES = 4
CLASS_NAMES = ["background", "onion", "seed", "stem"]

BATCH_SIZE = 2
LR = 6e-5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 50
PATIENCE = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. COCO SEGMENTATION -> BINARY MASK
# ------------------------------------------------------------
def decode_segmentation(segmentation, h, w):
    if not segmentation:
        return None

    if isinstance(segmentation, list):
        rles = maskUtils.frPyObjects(segmentation, h, w)
        rle = maskUtils.merge(rles)
        m = maskUtils.decode(rle)

    elif isinstance(segmentation, dict):
        if isinstance(segmentation.get("counts"), list):
            rle = maskUtils.frPyObjects(segmentation, h, w)
            m = maskUtils.decode(rle)
        else:
            m = maskUtils.decode(segmentation)
    else:
        return None

    if m.ndim == 3:
        m = np.any(m, axis=2)

    return m.astype(bool)

# Orijinal COCO category -> semantic class
# 1 onion_cell_single    -> 1 onion
# 2 onion_cells_multiple -> 1 onion
# 3 seed_tissue          -> 2 seed
# 4 stem_tissue          -> 3 stem
CATEGORY_MAP = {
    1: 1,
    2: 1,
    3: 2,
    4: 3
}

# ------------------------------------------------------------
# 3. SEMANTIC MASKLERİ OLUŞTUR
# ------------------------------------------------------------
for split in ["train", "valid", "test"]:

    split_dir = os.path.join(BASE, split)
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    out_dir = os.path.join(MASK_BASE, split)
    os.makedirs(out_dir, exist_ok=True)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    anns_by_img = defaultdict(list)

    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    for info in coco["images"]:

        h, w = info["height"], info["width"]
        semantic = np.zeros((h, w), dtype=np.uint8)

        for ann in anns_by_img[info["id"]]:

            cat = ann["category_id"]

            if cat not in CATEGORY_MAP:
                continue

            binary = decode_segmentation(
                ann.get("segmentation"),
                h,
                w
            )

            if binary is None:
                continue

            semantic[binary] = CATEGORY_MAP[cat]

        mask_name = os.path.splitext(info["file_name"])[0] + ".png"

        Image.fromarray(semantic).save(
            os.path.join(out_dir, mask_name)
        )

    print(split, "mask:", len(coco["images"]))

print("\nMaskeler hazır.")

# ------------------------------------------------------------
# 4. DATASET
# ------------------------------------------------------------
processor = SegformerImageProcessor.from_pretrained(
    MODEL_NAME,
    do_resize=True,
    size={"height": 512, "width": 512},
    do_reduce_labels=False
)

class MicroscopySegmentationDataset(Dataset):

    def __init__(self, split):

        self.split = split
        self.img_dir = os.path.join(BASE, split)
        self.mask_dir = os.path.join(MASK_BASE, split)

        ann_path = os.path.join(
            self.img_dir,
            "_annotations.coco.json"
        )

        with open(ann_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        self.images = coco["images"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        info = self.images[idx]

        image_path = os.path.join(
            self.img_dir,
            info["file_name"]
        )

        mask_name = (
            os.path.splitext(info["file_name"])[0]
            + ".png"
        )

        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        enc = processor(
            images=image,
            segmentation_maps=np.array(mask),
            return_tensors="pt"
        )

        return {
            "pixel_values": enc["pixel_values"].squeeze(0),
            "labels": enc["labels"].squeeze(0).long()
        }

train_ds = MicroscopySegmentationDataset("train")
valid_ds = MicroscopySegmentationDataset("valid")
test_ds  = MicroscopySegmentationDataset("test")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "\nDataset:",
    len(train_ds),
    len(valid_ds),
    len(test_ds)
)

# ------------------------------------------------------------
# 5. MODEL
# ------------------------------------------------------------
id2label = {
    0: "background",
    1: "onion",
    2: "seed",
    3: "stem"
}

label2id = {v: k for k, v in id2label.items()}

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(DEVICE)

print(
    "Parametre:",
    sum(p.numel() for p in model.parameters())
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# ------------------------------------------------------------
# 6. FOREGROUND MACRO DICE
# ------------------------------------------------------------
@torch.no_grad()
def validation_metrics():

    model.eval()

    cm = torch.zeros(
        NUM_CLASSES,
        NUM_CLASSES,
        dtype=torch.int64
    )

    losses = []

    for batch in valid_loader:

        x = batch["pixel_values"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            out = model(
                pixel_values=x,
                labels=y
            )

            loss = out.loss

        losses.append(loss.item())

        logits = F.interpolate(
            out.logits,
            size=y.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        pred = logits.argmax(1)

        yt = y.view(-1).cpu()
        yp = pred.view(-1).cpu()

        valid = (
            (yt >= 0)
            & (yt < NUM_CLASSES)
        )

        indices = (
            NUM_CLASSES * yt[valid]
            + yp[valid]
        )

        cm += torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        ).reshape(NUM_CLASSES, NUM_CLASSES)

    dices = []

    for c in range(1, NUM_CLASSES):

        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp

        dice = (
            2 * tp /
            (2 * tp + fp + fn + 1e-7)
        )

        dices.append(dice)

    return (
        float(np.mean(losses)),
        float(np.mean(dices)),
        dices
    )

# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------
best_dice = -1
best_epoch = -1
wait = 0

for epoch in range(1, MAX_EPOCHS + 1):

    model.train()

    train_losses = []

    for batch in train_loader:

        x = batch["pixel_values"].to(
            DEVICE,
            non_blocking=True
        )

        y = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            out = model(
                pixel_values=x,
                labels=y
            )

            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())

    val_loss, val_dice, class_dices = validation_metrics()

    print(
        f"Epoch {epoch:02d} | "
        f"Train {np.mean(train_losses):.4f} | "
        f"Val {val_loss:.4f} | "
        f"FG Dice {val_dice:.6f} | "
        f"Onion {class_dices[0]:.4f} | "
        f"Seed {class_dices[1]:.4f} | "
        f"Stem {class_dices[2]:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice
        best_epoch = epoch
        wait = 0

        checkpoint = {
            "epoch": epoch,
            "best_val_dice": best_dice,
            "model_name": MODEL_NAME,
            "class_names": CLASS_NAMES,
            "model_state_dict": model.state_dict()
        }

        torch.save(checkpoint, LOCAL_SAVE)
        torch.save(checkpoint, DRIVE_SAVE)

        print("  >>> BEST CHECKPOINT KAYDEDİLDİ")

    else:

        wait += 1

        if wait >= PATIENCE:
            print("\nEarly stopping.")
            break

print("\n================================")
print("SEGFORMER TAMAMLANDI")
print("Best epoch:", best_epoch)
print("Best validation Dice:", best_dice)
print("Local:", LOCAL_SAVE)
print("Drive:", DRIVE_SAVE)
print("================================")